<a href="https://colab.research.google.com/github/1900690/fisheye-distortion-correction/blob/main/%E3%82%BF%E3%82%A4%E3%83%A0%E3%83%A9%E3%83%97%E3%82%B9%E7%94%A8%E5%8B%95%E7%94%BB%E5%88%87%E3%82%8A%E5%8F%96%E3%82%8A%E8%A3%9C%E6%AD%A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. AVI動画のアップロード
import os
import cv2
import matplotlib.pyplot as plt
from google.colab import files

# フォルダの初期化（以前のデータをクリア）
for dir_name in ['/content/extracted_images', '/content/corrected_images']:
    !rm -rf {dir_name}
    os.makedirs(dir_name, exist_ok=True)

print("AVI動画ファイルを選択してください。")
uploaded = files.upload()

if uploaded:
    video_filename = list(uploaded.keys())[0]
    print(f"アップロード完了: {video_filename}")

    # 動画の最初のフレームを表示
    cap = cv2.VideoCapture(video_filename)
    if cap.isOpened():
        ret, first_frame = cap.read()
        if ret:
            print("\n【動画の最初のフレーム（確認用）】")
            plt.figure(figsize=(8, 6))
            plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()
        else:
            print("動画の最初のフレームを読み込めませんでした。")
        cap.release()
    else:
        print("動画ファイルを開けませんでした。")
else:
    print("アップロードがキャンセルされました。")

In [ ]:
#@title 2. 特定タイミングの画像抽出
import cv2
import os
import matplotlib.pyplot as plt

#@markdown 抽出の条件を指定してください。
撮影開始時間_動画先頭からの経過時間_時間単位 = 1.0 #@param {type:"number"}
抽出するフレーム数_枚数 = 1 #@param {type:"integer"}
切り取り間隔の時間_分単位 = 60 #@param {type:"number"}

if 'video_filename' not in locals():
    print("エラー: セル1で動画をアップロードしてください。")
else:
    # 抽出先フォルダを空にする
    !rm -rf /content/extracted_images/*

    cap = cv2.VideoCapture(video_filename)
    if not cap.isOpened():
        print("エラー: 動画を開けませんでした。")
    else:
        original_fps = cap.get(cv2.CAP_PROP_FPS)
        total_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"動画の元のFPS: {original_fps}")
        print(f"動画の総フレーム数: {total_video_frames}")

        # 開始フレーム位置と、指定間隔（分）ごとのフレーム数を計算
        start_frame = int(撮影開始時間_動画先頭からの経過時間_時間単位 * 3600 * original_fps)
        frame_interval = int(切り取り間隔の時間_分単位 * 60 * original_fps)

        if start_frame >= total_video_frames:
            print(f"エラー: 指定された撮影開始時間（{撮影開始時間_動画先頭からの経過時間_時間単位}時間後）は動画の長さを超えています。")
        else:
            extracted_count = 0
            first_image = None

            # 指定されたフレーム数分、間隔をあけて抽出
            for i in range(抽出するフレーム数_枚数):
                target_frame = start_frame + (i * frame_interval)

                # 動画の最大フレーム数を超えたら終了
                if target_frame >= total_video_frames:
                    print(f"警告: 指定されたフレーム数に達する前に動画の終端に達しました。({extracted_count}枚抽出済)")
                    break

                cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
                ret, frame = cap.read()
                if not ret:
                    break

                out_path = f"/content/extracted_images/frame_{target_frame:06d}.jpg"
                cv2.imwrite(out_path, frame)

                if first_image is None:
                    first_image = frame.copy()
                extracted_count += 1

            cap.release()
            print(f"抽出完了: 計 {extracted_count} 枚の画像を保存しました。")

            # 切り取り確認用に抽出した1枚目を表示
            if first_image is not None:
                print("\n【抽出画像の1枚目（切り取り確認用）】")
                plt.figure(figsize=(8, 6))
                plt.imshow(cv2.cvtColor(first_image, cv2.COLOR_BGR2RGB))
                plt.axis('off')
                plt.show()

#魚眼補正

###参考
[魚眼レンズの補正](https://qiita.com/hiro_o_tama/items/cb544fe64ca373750aae)


[OpenCVのundistort（レンズ歪み補正）で端っこが欠けてしまうのをなんとかする](https://qiita.com/jellied_unagi/items/36796d48d7d8a5fb3e42)


[端っこをかけるのを防ぐために](http://twinklesmile.blog42.fc2.com/blog-entry-466.html)

In [ ]:
#@title 3. 画像の魚眼補正（特定の機種のみ・必要な人のみ実行）
import cv2
import numpy as np
import os
import glob
import matplotlib.pyplot as plt

#@markdown **魚眼レンズ 補正パラメータ設定**
#@markdown * ※値はPythonの数式やリストの形式（例: [a, b, c]）で入力してください。
DIM_input = "(640, 480)" #@param {type:"string"}
K_input = "[[500.0, 0.0, 320.0], [0.0, 500.0, 240.0], [0.0, 0.0, 1.0]]" #@param {type:"string"}
D_input = "[-0.1, 0.01, 0.0, 0.0]" #@param {type:"string"}

!rm -rf /content/corrected_images/*
extracted_files = sorted(glob.glob('/content/extracted_images/*.jpg'))

if len(extracted_files) > 0:
    print("魚眼補正を実行しています...")
    try:
        # 入力文字列をPythonオブジェクトおよびnumpy配列に変換
        DIM = eval(DIM_input)  # (width, height) のタプル
        K = np.array(eval(K_input), dtype=np.float64)  # 3x3 カメラ行列
        D = np.array(eval(D_input), dtype=np.float64)  # 歪み係数

        # 補正マップの初期化 (fisheyeモデル用)
        map1, map2 = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), K, DIM, cv2.CV_16SC2)
        first_corrected = None

        for file_path in extracted_files:
            img = cv2.imread(file_path)

            # 画像サイズがDIMと異なる場合はDIMにリサイズ
            if (img.shape[1], img.shape[0]) != DIM:
                img = cv2.resize(img, DIM)

            # 魚眼補正の適用
            undistorted_img = cv2.remap(img, map1, map2, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)

            out_path = os.path.join('/content/corrected_images', os.path.basename(file_path))
            cv2.imwrite(out_path, undistorted_img)

            if first_corrected is None:
                first_corrected = undistorted_img.copy()

        print(f"補正完了: {len(extracted_files)} 枚の画像を処理しました。")

        if first_corrected is not None:
            print("\n【補正画像の1枚目（確認用）】")
            plt.figure(figsize=(8, 6))
            plt.imshow(cv2.cvtColor(first_corrected, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()

    except Exception as e:
        print(f"エラー: パラメータの解析または補正処理に失敗しました。入力形式を確認してください。\n詳細: {e}")
else:
    print("エラー: 補正する画像がありません。先にセル2を実行して画像を抽出してください。")

In [ ]:
#@title 4. 画像のまとめてダウンロード
import shutil
from google.colab import files
import os

#@markdown ダウンロードするZIPファイル名を指定してください。
zip_filename = "extracted_images" #@param {type:"string"}

corrected_dir = '/content/corrected_images'
extracted_dir = '/content/extracted_images'

# セル3で補正が実行され、画像が生成されているか判定
if os.path.exists(corrected_dir) and len(os.listdir(corrected_dir)) > 0:
    target_dir = corrected_dir
    print("魚眼補正済みの画像を圧縮しています...")
else:
    target_dir = extracted_dir
    print("抽出された元画像を圧縮しています（魚眼補正はスキップされました）...")

if not os.path.exists(target_dir) or len(os.listdir(target_dir)) == 0:
    print("エラー: ダウンロードする画像がありません。")
else:
    # フォルダをZIP化
    zip_path = f"/content/{zip_filename}"
    shutil.make_archive(zip_path, 'zip', root_dir=target_dir)

    # ZIPファイルをダウンロード
    print("ダウンロードを開始します...")
    files.download(f"{zip_path}.zip")